# 07 — Tokenización con HuggingFace

**Level 0 — Fundamentos Software & IA**

Cargamos un tokenizer real (BERT uncased), tokenizamos texto,
exploramos el vocabulario y vemos cómo funciona la subword tokenization (BPE).

In [1]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
print(f"   Tokenizer: {tokenizer.__class__.__name__}")
print(f"   Vocab size: {tokenizer.vocab_size:,} tokens")

/workspaces/student-ai/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


   Tokenizer: BertTokenizer
   Vocab size: 30,522 tokens


## Tokenizar texto básico

In [2]:
texto = "Hello, how are you?"
print(f"   Texto: '{texto}'")
tokens = tokenizer.tokenize(texto)
print(f"   Tokenize: {tokens}")
ids = tokenizer.encode(texto)
print(f"   Encode:   {ids}")
decodificado = tokenizer.decode(ids)
print(f"   Decode:   '{decodificado}'")

print("\n3. Mapeo token -> ID:")
for token, id_ in zip(tokens, ids):
    print(f"   '{token}' -> {id_}")

   Texto: 'Hello, how are you?'
   Tokenize: ['hello', ',', 'how', 'are', 'you', '?']
   Encode:   [101, 7592, 1010, 2129, 2024, 2017, 1029, 102]
   Decode:   '[CLS] hello, how are you? [SEP]'

3. Mapeo token -> ID:
   'hello' -> 101
   ',' -> 7592
   'how' -> 1010
   'are' -> 2129
   'you' -> 2024
   '?' -> 2017


## Subword tokenization (BPE)

In [3]:
palabras_prueba = [
    "unbelievably",
    "antidisestablishment",
    "tokenization",
    "chatbot",
    "LLM",
]
for palabra in palabras_prueba:
    tokens_palabra = tokenizer.tokenize(palabra)
    print(f"   '{palabra}' -> {tokens_palabra}")

   'unbelievably' -> ['un', '##bel', '##ie', '##va', '##bly']
   'antidisestablishment' -> ['anti', '##dis', '##est', '##ab', '##lish', '##ment']
   'tokenization' -> ['token', '##ization']
   'chatbot' -> ['chat', '##bot']
   'LLM' -> ['ll', '##m']


## Special tokens

In [4]:
print(f"   CLS: {tokenizer.cls_token} (id={tokenizer.cls_token_id})")
print(f"   SEP: {tokenizer.sep_token} (id={tokenizer.sep_token_id})")
print(f"   PAD: {tokenizer.pad_token} (id={tokenizer.pad_token_id})")
print(f"   UNK: {tokenizer.unk_token} (id={tokenizer.unk_token_id})")

   CLS: [CLS] (id=101)
   SEP: [SEP] (id=102)
   PAD: [PAD] (id=0)
   UNK: [UNK] (id=100)


## Padding y truncation

In [5]:
frases = ["Hola", "Hola mundo", "Hola mundo cruel"]
max_length = 5
print(f"   Max length: {max_length} tokens")
for frase in frases:
    encoded = tokenizer(
        frase,
        padding="max_length",
        truncation=True,
        max_length=max_length,
    )
    print(f"   '{frase}' -> ids={encoded['input_ids']}")

   Max length: 5 tokens
   'Hola' -> ids=[101, 7570, 2721, 102, 0]
   'Hola mundo' -> ids=[101, 7570, 2721, 25989, 102]
   'Hola mundo cruel' -> ids=[101, 7570, 2721, 25989, 102]


## Attention mask

In [6]:
encoded = tokenizer(
    ["Hola mundo", "Adios"],
    padding="max_length",
    max_length=5,
    return_tensors=None,
)
for i, frase in enumerate(["Hola mundo", "Adios"]):
    print(f"   '{frase}':")
    print(f"      input_ids:      {encoded['input_ids'][i]}")
    print(f"      attention_mask: {encoded['attention_mask'][i]}")

   'Hola mundo':
      input_ids:      [101, 7570, 2721, 25989, 102]
      attention_mask: [1, 1, 1, 1, 1]
   'Adios':
      input_ids:      [101, 27133, 2891, 102, 0]
      attention_mask: [1, 1, 1, 1, 0]


## Conclusión

- El tokenizer divide el texto en **subwords** (BPE), no en palabras
- Cada token tiene un **ID** del vocabulario; `encode`/`decode` son inversos
- **Special tokens** (CLS/SEP/PAD/UNK) marcan estructura del input
- **Padding + truncation** normalizan largos; el **attention mask** indica dónde hay texto real